# Credit Card Balance – Feature Engineering

Builds customer-level features from the `credit_card_balance` dataset for the Home Credit default prediction pipeline.

**Pipeline overview**
1. Load raw data
2. Impute missing values
3. Derive ratio features
4. Aggregate at previous-loan level & enrich with PrevLoans metadata
5. Monthly roll-up: sums, ratios, QoQ changes, max/DPD flags
6. Time-window aggregations (3Y, 1Y, 6M, 3M, last snapshot)
7. DPD timing & instalment event rate
8. Merge all features → prefix & save

## 1. Imports

In [2]:
import os
import pandas as pd
import numpy as np

## 2. Configuration & Paths

In [3]:
DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")

PATH_APP_TRAIN  = os.path.join(DATA_ROOT, "Input", "application_train.csv")
PATH_CC_BAL     = os.path.join(DATA_ROOT, "Input", "credit_card_balance.csv")
PATH_PREV_LOANS = os.path.join(DATA_ROOT, "Staging",
                               "prev_loan_data_processed.parquet")
PATH_OUTPUT     = os.path.join(DATA_ROOT, "Staging",
                               "creditcard_processed.parquet")

## 3. Load Data

In [4]:
base = pd.read_csv(PATH_APP_TRAIN, usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

PrevLoans = pd.read_parquet(PATH_PREV_LOANS)
PrevLoans = PrevLoans.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

CCBal = pd.read_csv(PATH_CC_BAL)
CCBal = CCBal.astype({'SK_ID_CURR': 'string', 'SK_ID_PREV': 'string'})

# Sort chronologically within each loan before any forward/back-fill
CCBal.sort_values(
    by=['SK_ID_CURR', 'SK_ID_PREV', 'MONTHS_BALANCE'],
    inplace=True, ignore_index=True,
)

# Restrict to training customers and drop TARGET (not used in feature engineering)
CCBal = base.merge(CCBal, on='SK_ID_CURR', how='inner').drop(columns=['TARGET'])

# Save unique customer IDs before any deletions – used as the merge spine later
unique_customers = CCBal[['SK_ID_CURR']].drop_duplicates()

print(f"CCBal shape   : {CCBal.shape}")
print(f"PrevLoans shape: {PrevLoans.shape}")

CCBal shape   : (3227965, 23)
PrevLoans shape: (1670214, 40)


- `AMT_BALANCE` - Balance during the month of previous credit
- `AMT_CREDIT_LIMIT_ACTUAL` - Credit card limit during the month of the previous credit
- `AMT_DRAWINGS_CURRENT` - Amount drawing during the month of the previous credit
- `AMT_PAYMENT_CURRENT`	- How much did the client pay during the month on the previous credit
- `AMT_PAYMENT_TOTAL_CURRENT` - How much did the client pay during the month in total on the previous credit
- `AMT_RECEIVABLE_PRINCIPAL` - Amount receivable for principal on the previous credit

## 4. Impute Missing Values

**Drawing sub-columns:** when `AMT_DRAWINGS_CURRENT == 0`, all sub-components
(ATM / POS / Other) must also be zero; any `NaN` there is a structural missing.

**Payment & min-regularity columns:** forward-fill then back-fill within each
loan (`SK_ID_PREV`), limited to 6 months in either direction.

In [4]:
# ── Structural zeros for drawing sub-columns ──────────────────────────────
drawing_sub_amt = [
    'AMT_DRAWINGS_ATM_CURRENT',
    'AMT_DRAWINGS_OTHER_CURRENT',
    'AMT_DRAWINGS_POS_CURRENT',
]
drawing_sub_cnt = [
    'CNT_DRAWINGS_ATM_CURRENT',
    'CNT_DRAWINGS_POS_CURRENT',
    'CNT_DRAWINGS_OTHER_CURRENT',
]
for col in drawing_sub_amt:
    CCBal.loc[(CCBal['AMT_DRAWINGS_CURRENT'] == 0) & CCBal[col].isna(), col] = 0
for col in drawing_sub_cnt:
    CCBal.loc[(CCBal['CNT_DRAWINGS_CURRENT'] == 0) & CCBal[col].isna(), col] = 0

# ── Time-series fill for payment columns (within each loan) ───────────────
for col in ['AMT_PAYMENT_CURRENT', 'AMT_INST_MIN_REGULARITY']:
    CCBal[col] = (
        CCBal.groupby('SK_ID_PREV')[col]
        .transform(lambda x: x.ffill(limit=6).bfill(limit=6))
    )

print("Remaining nulls in imputed columns:")
check_cols = drawing_sub_amt + drawing_sub_cnt + ['AMT_PAYMENT_CURRENT', 'AMT_INST_MIN_REGULARITY']
print(CCBal[check_cols].isna().sum().to_string())

Remaining nulls in imputed columns:
AMT_DRAWINGS_ATM_CURRENT           0
AMT_DRAWINGS_OTHER_CURRENT         0
AMT_DRAWINGS_POS_CURRENT           0
CNT_DRAWINGS_ATM_CURRENT           0
CNT_DRAWINGS_POS_CURRENT           0
CNT_DRAWINGS_OTHER_CURRENT         0
AMT_PAYMENT_CURRENT           554904
AMT_INST_MIN_REGULARITY       170343


## 5. Derive Ratio Features

All ratios use a `+1` denominator guard to avoid division by zero.
The helper `compute_ratios()` is reused later on the monthly-summed data.

In [5]:
def compute_ratios(df):
    """Add standard credit-card ratio columns to *df* in-place and return it."""
    df = df.copy()
    df['UTILIZATION_RATIO'] = np.where(
        df['AMT_CREDIT_LIMIT_ACTUAL'] >= 0,
        df['AMT_BALANCE'] / (1 + df['AMT_CREDIT_LIMIT_ACTUAL']) * 100,
        np.nan,
    )
    df['PAYMENT_RATIO'] = np.where(
        (df['AMT_PAYMENT_TOTAL_CURRENT'] + df['AMT_TOTAL_RECEIVABLE']) >= 0,
        df['AMT_PAYMENT_TOTAL_CURRENT']
        / (1 + df['AMT_PAYMENT_TOTAL_CURRENT'] + df['AMT_TOTAL_RECEIVABLE']) * 100,
        np.nan,
    )
    df['DRAWING_TO_PYMNT_RATIO'] = np.where(
        df['AMT_PAYMENT_TOTAL_CURRENT'] >= 0,
        df['AMT_DRAWINGS_CURRENT'] / (df['AMT_PAYMENT_TOTAL_CURRENT'] + 1) * 100,
        np.nan,
    )
    df['INT_COVERAGE_RATIO'] = np.where(
        df['AMT_PAYMENT_TOTAL_CURRENT'] >= 0,
        (df['AMT_TOTAL_RECEIVABLE'] - df['AMT_RECEIVABLE_PRINCIPAL'])
        / (df['AMT_PAYMENT_TOTAL_CURRENT'] + 1) * 100,
        np.nan,
    )
    df['POS_DRAWING_AMT_RATIO'] = np.where(
        df['AMT_DRAWINGS_CURRENT'] >= 0,
        df['AMT_DRAWINGS_POS_CURRENT'] / (1 + df['AMT_DRAWINGS_CURRENT']) * 100,
        np.nan,
    )
    df['ATM_DRAWING_AMT_RATIO'] = np.where(
        df['AMT_DRAWINGS_CURRENT'] >= 0,
        df['AMT_DRAWINGS_ATM_CURRENT'] / (1 + df['AMT_DRAWINGS_CURRENT']) * 100,
        np.nan,
    )
    df['POS_DRAWING_CNT_RATIO'] = np.where(
        df['CNT_DRAWINGS_CURRENT'] >= 0,
        df['CNT_DRAWINGS_POS_CURRENT'] / (1 + df['CNT_DRAWINGS_CURRENT']) * 100,
        np.nan,
    )
    df['ATM_DRAWING_CNT_RATIO'] = np.where(
        df['CNT_DRAWINGS_CURRENT'] >= 0,
        df['CNT_DRAWINGS_ATM_CURRENT'] / (1 + df['CNT_DRAWINGS_CURRENT']) * 100,
        np.nan,
    )
    return df


CCBal = compute_ratios(CCBal)
print("Ratio columns added:", [c for c in CCBal.columns if 'RATIO' in c])

Ratio columns added: ['UTILIZATION_RATIO', 'PAYMENT_RATIO', 'DRAWING_TO_PYMNT_RATIO', 'INT_COVERAGE_RATIO', 'POS_DRAWING_AMT_RATIO', 'ATM_DRAWING_AMT_RATIO', 'POS_DRAWING_CNT_RATIO', 'ATM_DRAWING_CNT_RATIO']


## 6. Aggregate at Previous-Loan Level

Two-step aggregation:
1. Collapse each `(SK_ID_CURR, SK_ID_PREV)` loan to a single row of summary stats.
2. Join `PrevLoans` metadata, then roll up to `SK_ID_CURR`.

In [ ]:
# ── Step 1: loan-level summary ────────────────────────────────────────────
CCBal_Prev_ID = (
    CCBal.groupby(['SK_ID_CURR', 'SK_ID_PREV'])
    .agg(
        # FIX: (x == 'Completed').any() already returns a scalar bool;
        # chaining .sum() on a bool was redundant. Use .astype(int) for clarity.
        COMPLETED_PREV_LOAN=(
            'NAME_CONTRACT_STATUS',
            lambda x: int((x == 'Completed').any()),
        ),
        AMT_CREDIT_LIMIT_ACTUAL_ORIGINAL_PREVLOAN=('AMT_CREDIT_LIMIT_ACTUAL', 'first'),
        AMT_CREDIT_LIMIT_ACTUAL_FINAL_PREVLOAN=('AMT_CREDIT_LIMIT_ACTUAL', 'last'),
        AMT_BALANCE_AVG_PREVLOAN=('AMT_BALANCE', 'mean'),
        AVG_UTILIZATION_RATIO_PREVLOAN=('UTILIZATION_RATIO', 'mean'),
        AVG_PAYMENT_RATIO_PREVLOAN=('PAYMENT_RATIO', 'mean'),
        AVG_DRAWING_TO_PYMNT_RATIO_PREVLOAN=('DRAWING_TO_PYMNT_RATIO', 'mean'),
        AVG_INT_COVERAGE_RATIO_PREVLOAN=('INT_COVERAGE_RATIO', 'mean'),
        AVG_POS_DRAWING_AMT_RATIO_PREVLOAN=('POS_DRAWING_AMT_RATIO', 'mean'),
        AVG_ATM_DRAWING_AMT_RATIO_PREVLOAN=('ATM_DRAWING_AMT_RATIO', 'mean'),
        AVG_POS_DRAWING_CNT_RATIO_PREVLOAN=('POS_DRAWING_CNT_RATIO', 'mean'),
        AVG_ATM_DRAWING_CNT_RATIO_PREVLOAN=('ATM_DRAWING_CNT_RATIO', 'mean'),
        DURATION_PREVLOAN=('MONTHS_BALANCE', lambda x: x.max() - x.min()),
    )
    .reset_index()
)

# ── Step 2: join PrevLoans metadata ───────────────────────────────────────
CCBal_Prev_ID = CCBal_Prev_ID.merge(
    PrevLoans, on=['SK_ID_CURR', 'SK_ID_PREV'], how='left'
)

# ── Step 3a: customer-level count features ─────────────────────────────────
CCBal_Prev_ID_agg_1 = CCBal_Prev_ID.groupby('SK_ID_CURR').agg(
    TOTAL_PREV_LOAN_CNT=('SK_ID_PREV', 'nunique'),
    COMPLETED_PREV_LOAN_CNT=('COMPLETED_PREV_LOAN', 'sum'),
    NFLAG_INSURED_ON_APPROVAL_PREVLOAN_CNT=('NFLAG_INSURED_ON_APPROVAL', 'sum'),
    DAYS_DECISION_PREVLOAN_AVG=('DAYS_DECISION', 'mean'),
    HIGH_YIELD_GROUP_CNT=('NAME_YIELD_GROUP', lambda x: (x == 'high').sum()),
    MIDDLE_YIELD_GROUP_CNT=('NAME_YIELD_GROUP', lambda x: (x == 'middle').sum()),
).reset_index()

# ── Step 3b: customer-level numeric features (max / mean / min) ───────────
features_to_aggregate = [
    'AMT_CREDIT_LIMIT_ACTUAL_ORIGINAL_PREVLOAN',
    'AMT_CREDIT_LIMIT_ACTUAL_FINAL_PREVLOAN',
    'AMT_BALANCE_AVG_PREVLOAN',
    'AVG_UTILIZATION_RATIO_PREVLOAN',
    'AVG_PAYMENT_RATIO_PREVLOAN',
    'AVG_DRAWING_TO_PYMNT_RATIO_PREVLOAN',
    'AVG_INT_COVERAGE_RATIO_PREVLOAN',
    'AVG_POS_DRAWING_AMT_RATIO_PREVLOAN',
    'AVG_ATM_DRAWING_AMT_RATIO_PREVLOAN',
    'AVG_POS_DRAWING_CNT_RATIO_PREVLOAN',
    'AVG_ATM_DRAWING_CNT_RATIO_PREVLOAN',
    'DURATION_PREVLOAN',
    'CREDIT_TO_APPL',
    'AMT_ANNUITY_TO_CREDIT_RATIO',
    'GOODSPRICE_TO_AMT',
]
CCBal_Prev_ID_agg_2 = (
    CCBal_Prev_ID
    .groupby('SK_ID_CURR')[features_to_aggregate]
    .agg(['max', 'mean'])
    .reset_index()   # FIX: was missing; SK_ID_CURR was left in the index
)
# Flatten MultiIndex columns produced by multi-function agg
CCBal_Prev_ID_agg_2.columns = [
    '_'.join(col).strip().upper() if col[1] else col[0].upper()
    for col in CCBal_Prev_ID_agg_2.columns
]

del CCBal_Prev_ID, PrevLoans, base
print(f"agg_1 shape: {CCBal_Prev_ID_agg_1.shape}")
print(f"agg_2 shape: {CCBal_Prev_ID_agg_2.shape}")

agg_1 shape: (86905, 7)
agg_2 shape: (86905, 46)


## 7. Monthly Roll-Up

When a customer holds multiple credit cards simultaneously, sum each metric
across cards to get a single row per `(SK_ID_CURR, MONTHS_BALANCE)`.
Ratios are then recomputed on the summed totals (not averaged across cards).
A binary `ISNULL_DRAWINGS_CURRENT_FLAG` marks months with zero drawings.

In [7]:
sum_cols = [
    'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL',
    'AMT_DRAWINGS_ATM_CURRENT', 'AMT_DRAWINGS_CURRENT',
    'AMT_DRAWINGS_OTHER_CURRENT', 'AMT_DRAWINGS_POS_CURRENT',
    'AMT_INST_MIN_REGULARITY', 'AMT_PAYMENT_CURRENT',
    'AMT_PAYMENT_TOTAL_CURRENT', 'AMT_RECEIVABLE_PRINCIPAL',
    'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE',
    'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT',
    'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT',
    'CNT_INSTALMENT_MATURE_CUM',
]

Sum_CCBal_Mon = (
    CCBal.groupby(['SK_ID_CURR', 'MONTHS_BALANCE'])[sum_cols]
    .sum()
    .reset_index()
)

# Recompute ratios on the summed (customer-month) totals
Sum_CCBal_Mon = compute_ratios(Sum_CCBal_Mon)
Sum_CCBal_Mon['ISNULL_DRAWINGS_CURRENT_FLAG'] = (
    Sum_CCBal_Mon['AMT_DRAWINGS_CURRENT'] == 0
).astype(int)

print(f"Sum_CCBal_Mon shape: {Sum_CCBal_Mon.shape}")

Sum_CCBal_Mon shape: (3222930, 28)


## 8. Quarter-over-Quarter (QoQ) Features

Compares each month's value to the value 3 months prior within the same customer:
- **Percentage change** for monetary / ratio columns (clipped to ≥ 0 before differencing)
- **Absolute change** for count columns

The first 3 months of each customer's history will be `NaN` (no prior quarter to compare).

In [8]:
def create_qoq_features(df, pct_cols, abs_cols):
    """
    Engineer Quarter-over-Quarter (QoQ) change features.

    Parameters
    ----------
    df : DataFrame
        Must contain 'SK_ID_CURR', 'MONTHS_BALANCE', and all columns in
        *pct_cols* and *abs_cols*. Rows must be sorted by
        ['SK_ID_CURR', 'MONTHS_BALANCE'] before calling.
    pct_cols : list of str
        Monetary / ratio columns — QoQ *percentage* change is computed.
        Values are clipped to 0 before differencing to avoid sign artefacts.
    abs_cols : list of str
        Count columns — QoQ *absolute* difference is computed.

    Returns
    -------
    DataFrame with SK_ID_CURR, MONTHS_BALANCE, and new *_QOQ_* columns only.
    """
    df = df.copy()
    grouped = df.groupby('SK_ID_CURR')

    df[pct_cols] = df[pct_cols].clip(lower=0)

    for col in pct_cols:
        prev_val = grouped[col].shift(3)
        df[f'{col}_QOQ_PCT_CHANGE'] = (
            (df[col] - prev_val)
            / prev_val.replace(0, np.nan)  # avoid ÷0; zero-to-positive → NaN
        ).replace([np.inf, -np.inf], np.nan)

    for col in abs_cols:
        df[f'{col}_QOQ_ABS_CHANGE'] = df[col] - grouped[col].shift(3)

    qoq_cols = [c for c in df.columns if '_QOQ_' in c]
    return df[['SK_ID_CURR', 'MONTHS_BALANCE'] + qoq_cols]


amount_cols = [
    'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL',
    'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_ATM_CURRENT',
    'AMT_DRAWINGS_POS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT',
    'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT',
    'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE',
    'UTILIZATION_RATIO', 'PAYMENT_RATIO',
    'DRAWING_TO_PYMNT_RATIO', 'INT_COVERAGE_RATIO',
    'POS_DRAWING_AMT_RATIO', 'ATM_DRAWING_AMT_RATIO',
    'POS_DRAWING_CNT_RATIO', 'ATM_DRAWING_CNT_RATIO',
]
other_cols = [
    'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT',
    'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT',
    'CNT_INSTALMENT_MATURE_CUM',
]

QoQ_CCBal_Mon = create_qoq_features(Sum_CCBal_Mon, amount_cols, other_cols)
print(f"QoQ features shape: {QoQ_CCBal_Mon.shape}")

QoQ features shape: (3222930, 26)


## 9. Monthly Maximum Values & DPD Flags

> **Fix:** the original aggregated `max_cols` from the raw `CCBal` (card-level rows).
> This is inconsistent with the customer-month level of `Sum_CCBal_Mon`.
> The max is now taken from `Sum_CCBal_Mon` so all monthly tables are at the
> same grain.

In [9]:
max_cols = [
    'SK_DPD', 'SK_DPD_DEF',
    'UTILIZATION_RATIO', 'PAYMENT_RATIO', 'DRAWING_TO_PYMNT_RATIO',
    'INT_COVERAGE_RATIO', 'POS_DRAWING_AMT_RATIO', 'ATM_DRAWING_AMT_RATIO',
    'POS_DRAWING_CNT_RATIO', 'ATM_DRAWING_CNT_RATIO',
]

# FIX: aggregate from CCBal (card-level) for DPD/SK_DPD_DEF which only exist
# there, but use Sum_CCBal_Mon for ratio columns that were already summed.
Max_CCBal_Mon = (
    CCBal.groupby(['SK_ID_CURR', 'MONTHS_BALANCE'])[['SK_DPD', 'SK_DPD_DEF']]
    .max()
    .reset_index()
)
# Bring ratio max from the already-summed monthly table
ratio_max_cols = [
    'UTILIZATION_RATIO', 'PAYMENT_RATIO', 'DRAWING_TO_PYMNT_RATIO',
    'INT_COVERAGE_RATIO', 'POS_DRAWING_AMT_RATIO', 'ATM_DRAWING_AMT_RATIO',
    'POS_DRAWING_CNT_RATIO', 'ATM_DRAWING_CNT_RATIO',
]
ratio_max = (
    Sum_CCBal_Mon.groupby(['SK_ID_CURR', 'MONTHS_BALANCE'])[ratio_max_cols]
    .max()
    .reset_index()
)
Max_CCBal_Mon = Max_CCBal_Mon.merge(
    ratio_max, on=['SK_ID_CURR', 'MONTHS_BALANCE'], how='left'
)
Max_CCBal_Mon.columns = [
    col + '_MAX' if col not in ('SK_ID_CURR', 'MONTHS_BALANCE') else col
    for col in Max_CCBal_Mon.columns
]

# DPD threshold flags
Max_CCBal_Mon['DPD_GE_1']  = (Max_CCBal_Mon['SK_DPD_MAX'] >= 1).astype(int)
Max_CCBal_Mon['DPD_GE_7']  = (Max_CCBal_Mon['SK_DPD_MAX'] >= 7).astype(int)
Max_CCBal_Mon['DPD_GE_30'] = (Max_CCBal_Mon['SK_DPD_MAX'] >= 30).astype(int)

print(f"Max_CCBal_Mon shape: {Max_CCBal_Mon.shape}")

Max_CCBal_Mon shape: (3222930, 15)


## 10. Merge Monthly Tables

Combine sums, QoQ changes, and max/DPD flags into one monthly table,
then define `agg_cols` (all feature columns excluding the keys).

In [10]:
CCBal_Mon = (
    Sum_CCBal_Mon
    .merge(QoQ_CCBal_Mon,  on=['SK_ID_CURR', 'MONTHS_BALANCE'], how='left')
    .merge(Max_CCBal_Mon,  on=['SK_ID_CURR', 'MONTHS_BALANCE'], how='left')
)

agg_cols = [c for c in CCBal_Mon.columns if c not in ('SK_ID_CURR', 'MONTHS_BALANCE')]

del Sum_CCBal_Mon, QoQ_CCBal_Mon, Max_CCBal_Mon
import gc; gc.collect()

print(f"CCBal_Mon shape: {CCBal_Mon.shape}")
print(f"Feature columns: {len(agg_cols)}")

CCBal_Mon shape: (3222930, 65)
Feature columns: 63


## 11. Time-Window Aggregations

Four windows computed independently. `std` is omitted for the 3-month window
because with only 3 data points it is rarely stable.

| Variable | Window | Suffix |
|---|---|---|
| `CCBal_3Y_Agg` | 3 years | `_LAST3Y` |
| `CCBal_2Y_Agg` | 2 years | `_LAST2Y` |
| `CCBal_1Y_Agg` | 1 year  | `_LAST1Y` |
| `CCBal_6M_Agg` | 6 months | `_LAST6M` |
| `CCBal_3M_Agg` | 3 months | `_LAST3M` |

<br>

> **Fix:** `reset_index()` is called *before* flattening the MultiIndex so
> `SK_ID_CURR` appears as a plain column with no trailing underscore. The
> original fragile guard `if col != 'SK_ID_CURR_'` is no longer needed.

In [ ]:
def build_window_agg(df, months_back, suffix, agg_funcs):
    """Aggregate agg_cols over the most recent *months_back* months."""
    window = df[df['MONTHS_BALANCE'] >= months_back]
    agg = (
        window.groupby('SK_ID_CURR')[agg_cols]
        .agg(agg_funcs)
        .reset_index()  # FIX: bring SK_ID_CURR out of index first
    )
    # Flatten MultiIndex: ('AMT_BALANCE', 'mean') → 'AMT_BALANCE_MEAN'
    agg.columns = [
        f"{col[0]}_{col[1]}{suffix}".upper() if col[1] else col[0]
        for col in agg.columns
    ]
    return agg


CCBal_3Y_Agg = build_window_agg(CCBal_Mon, -36, '_LAST3Y', ['mean','max','std'])
CCBal_2Y_Agg = build_window_agg(CCBal_Mon, -24, '_LAST2Y', ['mean','max','std'])
CCBal_1Y_Agg = build_window_agg(CCBal_Mon, -12, '_LAST1Y', ['mean','max','std'])
CCBal_6M_Agg = build_window_agg(CCBal_Mon,  -6, '_LAST6M', ['mean','max','std'])
CCBal_3M_Agg = build_window_agg(CCBal_Mon,  -3, '_LAST3M', ['mean','max'])

print("3Y:", CCBal_3Y_Agg.shape)
print("2Y:", CCBal_2Y_Agg.shape)
print("1Y:", CCBal_1Y_Agg.shape)
print("6M:", CCBal_6M_Agg.shape)
print("3M:", CCBal_3M_Agg.shape)

3Y: (86905, 253)
2Y: (86905, 253)
1Y: (86905, 253)
6M: (86905, 253)
3M: (85317, 190)


## 12. Last-Snapshot Features

The most recent month's values for each customer.

> **Fix:** `GroupBy.last()` is deprecated in pandas ≥ 2.0 (it skips `NaN`,
> not the same as the literal last row). Replaced with `nth(-1)` which
> returns the actual last row regardless of `NaN`.

`MONTHS_BALANCE` of the last record is renamed `DAYS_SINCE_LAST_TRANSACTION`
and negated to express it as a positive count of months since the last activity.

In [15]:
temp = CCBal_Mon.groupby('SK_ID_CURR')[['SK_ID_CURR','MONTHS_BALANCE'] + agg_cols ].nth(-1).reset_index()

In [18]:
# FIX: use .nth(-1) instead of deprecated .last()
CCBal_Last_Snapshot = (
    CCBal_Mon.sort_values(['SK_ID_CURR', 'MONTHS_BALANCE'])
    .groupby('SK_ID_CURR')[['SK_ID_CURR', 'MONTHS_BALANCE'] + agg_cols ]
    .nth(-1)
    .reset_index()
    .drop(columns=['index'])  # drop the extra index column added by reset_index(
)
CCBal_Last_Snapshot.columns = [
    col + '_LAST_MONTH' if col != 'SK_ID_CURR' else col
    for col in CCBal_Last_Snapshot.columns
]
CCBal_Last_Snapshot.rename(
    columns={'MONTHS_BALANCE_LAST_MONTH': 'DAYS_SINCE_LAST_TRANSACTION'},
    inplace=True,
)
# Negate: MONTHS_BALANCE is negative; flip to positive 'months ago'
CCBal_Last_Snapshot['DAYS_SINCE_LAST_TRANSACTION'] = (
    -CCBal_Last_Snapshot['DAYS_SINCE_LAST_TRANSACTION']
)

print(f"Last-snapshot shape: {CCBal_Last_Snapshot.shape}")

Last-snapshot shape: (86905, 65)


## 13. DPD Timing Feature

`DAYS_TO_DPD_GE1` = number of months from a customer's *first ever credit-card
record* to their *first DPD ≥ 1 event*, converted to approximate days (× 30).

Because `MONTHS_BALANCE` is sorted ascending (most-negative = oldest), `.first()`
correctly retrieves the earliest occurrence in each group.
Customers with no DPD event receive `NaN`.

In [19]:
# Earliest DPD ≥ 1 month for each customer
CCBal_DPDge1 = (
    CCBal_Mon[CCBal_Mon['SK_DPD_MAX'] >= 1]
    .groupby('SK_ID_CURR')[['MONTHS_BALANCE']]
    .first()
    .reset_index()
    .rename(columns={'MONTHS_BALANCE': 'MONTH_DPD_GE1'})
)

# Earliest month on record per customer (the credit-history start)
CCBal_First_Snapshot = (
    CCBal_Mon.groupby('SK_ID_CURR')[['MONTHS_BALANCE']]
    .first()
    .reset_index()
)

CCBal_DPDge1 = CCBal_DPDge1.merge(
    CCBal_First_Snapshot, on='SK_ID_CURR', how='left'
)
# Both values are negative; the difference (DPD_month - start_month) is ≥ 0
CCBal_DPDge1['DAYS_TO_DPD_GE1'] = (
    (CCBal_DPDge1['MONTH_DPD_GE1'] - CCBal_DPDge1['MONTHS_BALANCE']) * 30
)
CCBal_DPDge1 = CCBal_DPDge1[['SK_ID_CURR', 'DAYS_TO_DPD_GE1']]

print(f"Customers with DPD ≥ 1 event: {len(CCBal_DPDge1):,}")

Customers with DPD ≥ 1 event: 17,882


## 14. Instalment Event Rate

`INSTALLMENT_EVENT_RATE_AVG` = fraction of months in which the cumulative
instalment count increased (i.e., an instalment was actually paid that month).
A higher rate indicates more consistent repayment behaviour.

In [20]:
CCBal_installment_feature = (
    CCBal_Mon.groupby('SK_ID_CURR')['CNT_INSTALMENT_MATURE_CUM']
    .apply(lambda x: (x.diff() > 0).mean())
    .reset_index(name='INSTALLMENT_EVENT_RATE_AVG')
)

del CCBal_Mon, CCBal
import gc; gc.collect()
print("Monthly tables released.")

Monthly tables released.


## 15. Merge All Features

The merge spine is `unique_customers` (saved at load time).

> **Fix:** the original used `CCBal['SK_ID_CURR']` here — but `CCBal` was
> deleted two cells earlier, causing a `NameError` at runtime.

Activity-window flags (`IS_IN_LAST_*`) are added via a helper that uses
the merge indicator rather than a verbose `lambda`.

In [21]:
def flag_merge(left, right, on, flag_col):
    """Left-merge and record whether the row matched (1) or not (0)."""
    merged = left.merge(right, on=on, how='left', indicator=True)
    merged[flag_col] = (merged['_merge'] == 'both').astype(int)
    return merged.drop(columns=['_merge'])


# FIX: use unique_customers saved at load time (CCBal was deleted above)
final_cc_aggregated = unique_customers.copy()

# Previous-loan features (no activity flag needed – always present if customer has CC history)
final_cc_aggregated = final_cc_aggregated.merge(CCBal_Prev_ID_agg_1, on='SK_ID_CURR', how='left')
final_cc_aggregated = final_cc_aggregated.merge(CCBal_Prev_ID_agg_2, on='SK_ID_CURR', how='left')

# Time-window features with activity flags
final_cc_aggregated = flag_merge(final_cc_aggregated, CCBal_3Y_Agg, 'SK_ID_CURR', 'IS_IN_LAST_3Y')
final_cc_aggregated = flag_merge(final_cc_aggregated, CCBal_1Y_Agg, 'SK_ID_CURR', 'IS_IN_LAST_1Y')
final_cc_aggregated = flag_merge(final_cc_aggregated, CCBal_6M_Agg, 'SK_ID_CURR', 'IS_IN_LAST_6M')
final_cc_aggregated = flag_merge(final_cc_aggregated, CCBal_3M_Agg, 'SK_ID_CURR', 'IS_IN_LAST_3M')

# Snapshot, DPD timing, instalment rate
final_cc_aggregated = final_cc_aggregated.merge(CCBal_Last_Snapshot,       on='SK_ID_CURR', how='left')
final_cc_aggregated = final_cc_aggregated.merge(CCBal_DPDge1,              on='SK_ID_CURR', how='left')
final_cc_aggregated = final_cc_aggregated.merge(CCBal_installment_feature, on='SK_ID_CURR', how='left')

print(f"Final shape: {final_cc_aggregated.shape}")
final_cc_aggregated.head(3)

Final shape: (86905, 1067)


,SK_ID_CURR,TOTAL_PREV_LOAN_CNT,COMPLETED_PREV_LOAN_CNT,NFLAG_INSURED_ON_APPROVAL_PREVLOAN_CNT,DAYS_DECISION_PREVLOAN_AVG,HIGH_YIELD_GROUP_CNT,MIDDLE_YIELD_GROUP_CNT,AMT_CREDIT_LIMIT_ACTUAL_ORIGINAL_PREVLOAN_MAX,AMT_CREDIT_LIMIT_ACTUAL_ORIGINAL_PREVLOAN_MEAN,AMT_CREDIT_LIMIT_ACTUAL_ORIGINAL_PREVLOAN_MIN,...,INT_COVERAGE_RATIO_MAX_LAST_MONTH,POS_DRAWING_AMT_RATIO_MAX_LAST_MONTH,ATM_DRAWING_AMT_RATIO_MAX_LAST_MONTH,POS_DRAWING_CNT_RATIO_MAX_LAST_MONTH,ATM_DRAWING_CNT_RATIO_MAX_LAST_MONTH,DPD_GE_1_LAST_MONTH,DPD_GE_7_LAST_MONTH,DPD_GE_30_LAST_MONTH,DAYS_TO_DPD_GE1,INSTALLMENT_EVENT_RATE_AVG
0,100006,1,0,0.0,-181.0,0,0,270000,270000.0,270000,...,0.0,0.0,0.0,0.0,0.0,0,0,0,NaN,0.000000
1,100011,1,0,0.0,-2279.0,0,0,180000,180000.0,180000,...,0.0,0.0,0.0,0.0,0.0,0,0,0,NaN,0.445946
2,100021,1,1,0.0,-549.0,0,0,675000,675000.0,675000,...,0.0,0.0,0.0,0.0,0.0,0,0,0,NaN,0.000000


## 16. Prefix Columns & Quality Checks

In [22]:
# Prefix all feature columns with CREDITCARD_
final_cc_aggregated.columns = [
    'CREDITCARD_' + col if col != 'SK_ID_CURR' else col
    for col in final_cc_aggregated.columns
]

# Null rates
null_rates = final_cc_aggregated.isna().mean().sort_values(ascending=False)
print("Null rate (top 15 columns):")
print(null_rates.head(15).to_string())

# No duplicate customers
n_dupes = final_cc_aggregated['SK_ID_CURR'].duplicated().sum()
assert n_dupes == 0, f"Duplicate SK_ID_CURR found: {n_dupes}"
print(f"\nNo duplicate SK_ID_CURR. Final row count: {len(final_cc_aggregated):,}")

Null rate (top 15 columns):
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_LAST_MONTH     0.998585
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_STD_LAST6M     0.998147
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_STD_LAST1Y     0.996157
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MAX_LAST3M     0.995639
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MIN_LAST3M     0.995639
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MEAN_LAST3M    0.995639
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_STD_LAST3Y     0.991911
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MEAN_LAST6M    0.991174
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MIN_LAST6M     0.991174
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MAX_LAST6M     0.991174
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MAX_LAST1Y     0.984190
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MEAN_LAST1Y    0.984190
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CH

## 17. Save Output

In [23]:
final_cc_aggregated.to_parquet(PATH_OUTPUT, index=False)
print(f"Saved → {PATH_OUTPUT}")

Saved → a:\Dominic\My Projects\Home Credit\src\intermediate\creditcard_processed.parquet


## 18. Final Cleanup

In [24]:
del (
    CCBal_installment_feature, CCBal_DPDge1, CCBal_Last_Snapshot,
    CCBal_3M_Agg, CCBal_6M_Agg, CCBal_1Y_Agg, CCBal_3Y_Agg,
    CCBal_Prev_ID_agg_2, CCBal_Prev_ID_agg_1,
    unique_customers,
)
import gc; gc.collect()
print("All DataFrames released.")

All DataFrames released.
